# PDDL Planning Copilot — single-model run on Colab / Kaggle

Drives `run_experiment.py` from a Jupyter notebook on a free-tier T4. Auto-detects Colab vs Kaggle vs local; persists results and the run log to durable storage (Google Drive on Colab, `/kaggle/working/` on Kaggle); resumes from `trials.jsonl` if the session disconnects.

**Defaults are tuned for free-tier wall-clock budgets**: one small cluster-roster model (`Qwen3.5:0.8B`), `--partial 2`, `--num-variants 1`. Edit the parameters cell to swap in `qwen3.6:27b` (the larger cluster-roster model — needs Colab Pro / A100 or Kaggle dual-T4; a 27B model will not fit a single free T4).

Reproduces a subset of [Benyamin et al., 2025 (arXiv:2509.12987)](https://arxiv.org/abs/2509.12987). For the full sweep, run on the BGU cluster (see `cluster-experimenting/`) or a laptop (`./run_background.sh`).

In [ ]:
# Parameters — papermill-friendly. Edit and re-run.
MODEL         = "Qwen3.5:0.8B"  # cluster-roster set: Qwen3.5:0.8B (free T4 OK) | qwen3.6:27b (needs A100 / dual-T4)
THINK         = "default"       # default | on | off
PARTIAL_K     = 2               # 0 = full set; 2 = first-2 fixtures per domain (fast slice)
NUM_VARIANTS  = 1               # 1..3 (paper used 5; 1 keeps wall-clock down)
CONDITIONS    = "both"          # both | tools | no-tools
CONCURRENCY   = 2               # paired with OLLAMA_NUM_PARALLEL
RUN_TAG       = ""              # auto-stamped if empty
RESULTS_BASE  = ""              # auto: Drive on Colab, /kaggle/working/results on Kaggle
WORK_DIR      = ""              # auto: /content (Colab), /kaggle/working/repos (Kaggle)

In [ ]:
import os, sys, time, subprocess, shlex, json
from pathlib import Path

def _detect_platform():
    if "google.colab" in sys.modules:
        return "colab"
    if Path("/kaggle").exists():
        return "kaggle"
    return "local"

PLATFORM = _detect_platform()
print(f"Platform: {PLATFORM}")

if not WORK_DIR:
    WORK_DIR = {
        "colab":  "/content",
        "kaggle": "/kaggle/working/repos",
        "local":  str(Path.cwd().parent),
    }[PLATFORM]
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

if not RESULTS_BASE:
    if PLATFORM == "colab":
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        RESULTS_BASE = "/content/drive/MyDrive/pddl-copilot-experiments/results"
    elif PLATFORM == "kaggle":
        RESULTS_BASE = "/kaggle/working/results"
    else:
        RESULTS_BASE = str(Path(WORK_DIR) / "pddl-copilot-experiments" / "results" / "notebook")

if not RUN_TAG:
    safe_model = MODEL.replace(":", "-").replace("/", "-")
    RUN_TAG = f"{safe_model}_{time.strftime('%Y%m%d_%H%M%S')}"

OUTPUT_DIR = str(Path(RESULTS_BASE) / RUN_TAG)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"WORK_DIR    = {WORK_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")

In [ ]:
# Java 17+ is required by ENHSP (numeric_planner). Colab and Kaggle ship
# OpenJDK 11 by default; install 17 idempotently.
def sh(cmd, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, text=True).returncode

if PLATFORM in ("colab", "kaggle"):
    sh("apt-get -qq update")
    sh("apt-get -qq install -y openjdk-17-jre-headless")
    # Make Java 17 the default if the system has multiple JREs.
    sh("update-java-alternatives -s java-1.17.0-openjdk-amd64 2>/dev/null || true", check=False)
sh("java -version")

In [ ]:
# Clone both repos under WORK_DIR (idempotent).
EXPT_DIR        = Path(WORK_DIR) / "pddl-copilot-experiments"
MARKETPLACE_DIR = Path(WORK_DIR) / "pddl-copilot"

def _clone(url, dst):
    if dst.exists():
        print(f"exists: {dst}")
        return
    sh(f"git clone --depth 1 {url} {shlex.quote(str(dst))}")

_clone("https://github.com/SPL-BGU/pddl-copilot-experiments.git", EXPT_DIR)
_clone("https://github.com/SPL-BGU/pddl-copilot.git",            MARKETPLACE_DIR)

sh(f"pip install --quiet -r {shlex.quote(str(EXPT_DIR / 'requirements.txt'))}")

In [ ]:
# Pre-create plugin venvs so the first MCP spawn is instant
# (mirrors cluster-experimenting/setup_env.sh).
for plugin in ("pddl-solver", "pddl-validator"):
    plug_dir = MARKETPLACE_DIR / "plugins" / plugin
    venv = plug_dir / ".venv"
    if venv.exists():
        print(f"{plugin}: .venv exists")
        continue
    print(f"{plugin}: creating .venv...")
    sh(f"python3 -m venv {shlex.quote(str(venv))}")
    sh(f"{shlex.quote(str(venv / 'bin' / 'pip'))} install --quiet --upgrade pip")
    sh(f"{shlex.quote(str(venv / 'bin' / 'pip'))} install --quiet -r {shlex.quote(str(plug_dir / 'requirements.txt'))}")

In [ ]:
# Install Ollama, start `ollama serve` in the background, pull the model.
import urllib.request, urllib.error

if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
    sh("curl -fsSL https://ollama.com/install.sh | sh")

os.environ["OLLAMA_NUM_PARALLEL"] = str(CONCURRENCY)
os.environ.setdefault("OLLAMA_KEEP_ALIVE", "5m")

def _ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not _ollama_up():
    log_path = Path(OUTPUT_DIR) / "ollama_serve.log"
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(log_path, "w"),
                     stderr=subprocess.STDOUT,
                     env={**os.environ})
    for _ in range(30):
        if _ollama_up():
            break
        time.sleep(1)
    if not _ollama_up():
        raise RuntimeError(f"ollama serve did not become ready; see {log_path}")

print("ollama up")
sh(f"ollama pull {shlex.quote(MODEL)}")
sh("ollama list")

In [ ]:
# Run run_experiment.py. Stream stdout to the cell AND tee to OUTPUT_DIR/run.log
# so the log persists with the rest of the run artefacts on Drive / kaggle output.
think_args   = [] if THINK == "default" else ["--think", THINK]
partial_args = [] if PARTIAL_K <= 0  else ["--partial", str(PARTIAL_K)]

cmd = [
    "python3", "run_experiment.py",
    "--marketplace-path", str(MARKETPLACE_DIR),
    "--models",            MODEL,
    "--conditions",        CONDITIONS,
    "--num-variants",      str(NUM_VARIANTS),
    "--concurrency",       str(CONCURRENCY),
    "--output-dir",        OUTPUT_DIR,
    *think_args, *partial_args,
]
print("$ " + " ".join(shlex.quote(c) for c in cmd))

run_log = Path(OUTPUT_DIR) / "run.log"
proc = subprocess.Popen(cmd, cwd=str(EXPT_DIR),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
with open(run_log, "w") as logf:
    for line in proc.stdout:
        sys.stdout.write(line)
        logf.write(line)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"run_experiment.py exited with {rc}; see {run_log}")
print(f"\nDone. Results: {OUTPUT_DIR}")

In [ ]:
# Summary: list artefacts and print per-(model, condition, task) pass rates
# from the latest summary_*.json. Schema: see pddl_eval/summary.py.
print("Output files:")
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size:>10} bytes")

summary_files = sorted(Path(OUTPUT_DIR).glob("summary_*.json"))
if summary_files:
    s = json.loads(summary_files[-1].read_text())
    print(f"\nSummary: {summary_files[-1].name}")
    print(f"  meta.think      = {s['meta'].get('think')}")
    print(f"  meta.conditions = {s['meta'].get('conditions')}")
    print(f"  meta.models     = {s['meta'].get('models')}")
    print()
    print(f"  {'model':22s} {'cond':10s} {'task':18s} {'succ/n':>8}  rate (95% Wilson CI)")
    for r in s.get("single_task", []):
        rate = r.get("success_rate")
        lo, hi = r.get("ci_lo"), r.get("ci_hi")
        sn = f"{r.get('successes')}/{r.get('n')}"
        rate_str = "n/a" if rate is None else f"{rate:.3f} [{lo:.3f}, {hi:.3f}]"
        print(f"  {r['model']:22s} {r['condition']:10s} {r['task']:18s} {sn:>8}  {rate_str}")
else:
    print("\nNo summary_*.json found \u2014 inspect run.log for the failure mode.")

## Notes

- **Resume**: if the session dies mid-run, re-execute every cell. `run_experiment.py` reads `OUTPUT_DIR/trials.jsonl` and skips already-completed trials, so only the in-flight trial is re-run. To start fresh, append `--no-resume` in the run cell or delete `OUTPUT_DIR`.
- **Scaling up**: edit the parameters cell. `PARTIAL_K=0` runs the full fixture set; `NUM_VARIANTS=3` runs all active prompt variants; bump `MODEL` to `qwen3.6:27b` for the larger cluster-roster model (won't fit a single free T4 — use Colab Pro / A100 / L4, or Kaggle dual-T4 with Ollama auto-sharding).
- **Sharding across sessions**: pass `--shard i/N` (where `0 ≤ i < N`) by appending it to `cmd` in the run cell. Each shard hits a deterministic SHA-256 partition of the trial keys.
- **Kaggle persistence**: `/kaggle/working/` is saved with the notebook's output. To carry results across sessions, snapshot it as a Kaggle Dataset from the right-hand panel.
- **Where this fits**: see `EXPERIMENTS_FLOW.md` for methodology and `.claude/skills/analyzer/SKILL.md` for the aggregation/plotting skill that reads the produced `summary_*.json` and `trials.jsonl`.